In [2]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-GN-8B-PN-GN-20-43ae4354",  # Your fine-tuned protein model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in gene nomenclature and protein names. Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since gene symbols are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"gene_symbol": "Error"})

# Improved function to extract JSON from the LLM response for gene data
def extract_gene_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'gene_symbol' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'gene_symbol' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"gene_symbol"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'gene_symbol' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract gene_symbol value directly
        gene_symbol_match = re.search(r'"gene_symbol"[\s:]*"([^"]+)"', response_text)
        if gene_symbol_match:
            return {"gene_symbol": gene_symbol_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        gene_symbol_match = re.search(r'gene_symbol["\':\s]+([^"\'}\s,]+)', response_text)
        if gene_symbol_match:
            return {"gene_symbol": gene_symbol_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"gene_symbol": "None"}

        # If all parsing attempts fail
        return {"gene_symbol": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"gene_symbol": "Parse_Error"}

# Enhanced prompt to get gene symbol for a given protein name
def get_gene_symbol_from_protein(protein_name):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the protein name "{protein_name}", provide the corresponding gene symbol.

Instructions:
- Provide the EXACT official gene symbol (HUGO Gene Nomenclature Committee approved symbol)
- Choose the most direct, standard gene symbol for the protein
- Avoid gene aliases or alternative names (e.g., prefer "TP53" over "P53")
- Return the primary, commonly used gene symbol
- Gene symbols are typically short, uppercase abbreviations
- Examples: 
  - "p53" → "TP53"
  - "Breast cancer type 1 susceptibility protein" → "BRCA1"
  - "Insulin" → "INS"
- If the protein name is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "gene_symbol": "<gene_symbol_here>"
}}

Protein Name: {protein_name}"""

    response_text = query_together(prompt)
    result = extract_gene_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "protein_name": protein_name,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("protein_to_gene_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("gene_symbol", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_gene_symbol, new_gene_symbol):
    """
    Calculate match result between original and new gene symbols
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_gene_symbol) or pd.isna(new_gene_symbol):
        return 0
    return int(str(original_gene_symbol).strip() == str(new_gene_symbol).strip())

# Main processing function
def main():
    print("Starting protein name to gene symbol mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("protein_to_gene_api_responses_finetuned.jsonl"):
        with open("protein_to_gene_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample protein name
    print("Testing API connection...")
    test_result = get_gene_symbol_from_protein("p53")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the protein names CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_gene_symbol"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            protein_name = row["protein_name"]  # Using the protein_name column from the CSV
            original_gene_symbol = row["GN"]  # Original gene symbol for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {protein_name}")

            start_time = time.time()
            new_gene_symbol = get_gene_symbol_from_protein(protein_name)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new gene symbol
            df.loc[idx, "new_gene_symbol"] = new_gene_symbol
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_gene_symbol, new_gene_symbol)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_gene_symbol}, New: {new_gene_symbol}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_protein_to_gene_results_progress20.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} protein names.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_gene_symbol"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_protein_to_gene_results_final20.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Protein to Gene) ===")
        print(f"Total protein names processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_gene_symbol"] == "None").sum()
        error_results = (df["new_gene_symbol"] == "Error").sum()
        parse_error_results = (df["new_gene_symbol"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid gene symbols returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_gene_symbol"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about bins if available
        if 'bin' in df.columns:
            print(f"\nBreakdown by Bin:")
            bin_stats = df.groupby('bin').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(bin_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_protein_to_gene_results_final5.csv")
    print(f"Progress file: finetuned_protein_to_gene_results_progress5.csv")
    print(f"Log file: protein_to_gene_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_protein, normalized_gene_name, Match_GN")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting protein name to gene symbol mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Test result: TP53
API test successful, proceeding with batch processing...
Loaded 3978 rows from C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv
Columns in the dataset: ['ID', 'AC', 'GN', 'protein_name', 'GO_F Count', 'GO_P Count', 'GO_C Count', 'go_counts', 'bin', 'normalized_protein', 'normalized_gene_name', 'Match_GN', 'pmc_GN', 'PMC_protein_name']
Processing 1/3978: Exonuclease 3'-5' domain-containing protein 2 
Sending request (attempt 1/5)...
  Original: EXD2, New: EXO3, Match: 0
Progress saved. Processed 1/3978 protein names.
Waiting 3.98 seconds before next request...
Processing 2/3978: Exosome complex component MTR3
Sending request (attempt 1/5)...
  Original: EXOSC6, New: EXOSC3, Match: 0
Waiting 2.94 seconds before next request...
Processing 3/3978: Innate immunity acti

15 eps

In [ ]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-Protein-8B-Protein-Gene-1-70326a78",  # Your fine-tuned protein model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in gene nomenclature and protein names. Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since gene symbols are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"gene_symbol": "Error"})

# Improved function to extract JSON from the LLM response for gene data
def extract_gene_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'gene_symbol' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'gene_symbol' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"gene_symbol"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'gene_symbol' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract gene_symbol value directly
        gene_symbol_match = re.search(r'"gene_symbol"[\s:]*"([^"]+)"', response_text)
        if gene_symbol_match:
            return {"gene_symbol": gene_symbol_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        gene_symbol_match = re.search(r'gene_symbol["\':\s]+([^"\'}\s,]+)', response_text)
        if gene_symbol_match:
            return {"gene_symbol": gene_symbol_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"gene_symbol": "None"}

        # If all parsing attempts fail
        return {"gene_symbol": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"gene_symbol": "Parse_Error"}

# Enhanced prompt to get gene symbol for a given protein name
def get_gene_symbol_from_protein(protein_name):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the protein name "{protein_name}", provide the corresponding gene symbol.

Instructions:
- Provide the EXACT official gene symbol (HUGO Gene Nomenclature Committee approved symbol)
- Choose the most direct, standard gene symbol for the protein
- Avoid gene aliases or alternative names (e.g., prefer "TP53" over "P53")
- Return the primary, commonly used gene symbol
- Gene symbols are typically short, uppercase abbreviations
- Examples: 
  - "p53" → "TP53"
  - "Breast cancer type 1 susceptibility protein" → "BRCA1"
  - "Insulin" → "INS"
- If the protein name is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "gene_symbol": "<gene_symbol_here>"
}}

Protein Name: {protein_name}"""

    response_text = query_together(prompt)
    result = extract_gene_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "protein_name": protein_name,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("protein_to_gene_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("gene_symbol", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_gene_symbol, new_gene_symbol):
    """
    Calculate match result between original and new gene symbols
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_gene_symbol) or pd.isna(new_gene_symbol):
        return 0
    return int(str(original_gene_symbol).strip() == str(new_gene_symbol).strip())

# Main processing function
def main():
    print("Starting protein name to gene symbol mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("protein_to_gene_api_responses_finetuned.jsonl"):
        with open("protein_to_gene_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample protein name
    print("Testing API connection...")
    test_result = get_gene_symbol_from_protein("p53")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the protein names CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_gene_symbol"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            protein_name = row["protein_name"]  # Using the protein_name column from the CSV
            original_gene_symbol = row["GN"]  # Original gene symbol for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {protein_name}")

            start_time = time.time()
            new_gene_symbol = get_gene_symbol_from_protein(protein_name)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new gene symbol
            df.loc[idx, "new_gene_symbol"] = new_gene_symbol
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_gene_symbol, new_gene_symbol)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_gene_symbol}, New: {new_gene_symbol}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_protein_to_gene_results_progress15.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} protein names.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_gene_symbol"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_protein_to_gene_results_final15.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Protein to Gene) ===")
        print(f"Total protein names processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_gene_symbol"] == "None").sum()
        error_results = (df["new_gene_symbol"] == "Error").sum()
        parse_error_results = (df["new_gene_symbol"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid gene symbols returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_gene_symbol"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about bins if available
        if 'bin' in df.columns:
            print(f"\nBreakdown by Bin:")
            bin_stats = df.groupby('bin').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(bin_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_protein_to_gene_results_final5.csv")
    print(f"Progress file: finetuned_protein_to_gene_results_progress5.csv")
    print(f"Log file: protein_to_gene_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_protein, normalized_gene_name, Match_GN")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

10eps

In [ ]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-Protein-8B-Protein-Gene-1-70326a78",  # Your fine-tuned protein model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in gene nomenclature and protein names. Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since gene symbols are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"gene_symbol": "Error"})

# Improved function to extract JSON from the LLM response for gene data
def extract_gene_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'gene_symbol' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'gene_symbol' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"gene_symbol"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'gene_symbol' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract gene_symbol value directly
        gene_symbol_match = re.search(r'"gene_symbol"[\s:]*"([^"]+)"', response_text)
        if gene_symbol_match:
            return {"gene_symbol": gene_symbol_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        gene_symbol_match = re.search(r'gene_symbol["\':\s]+([^"\'}\s,]+)', response_text)
        if gene_symbol_match:
            return {"gene_symbol": gene_symbol_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"gene_symbol": "None"}

        # If all parsing attempts fail
        return {"gene_symbol": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"gene_symbol": "Parse_Error"}

# Enhanced prompt to get gene symbol for a given protein name
def get_gene_symbol_from_protein(protein_name):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the protein name "{protein_name}", provide the corresponding gene symbol.

Instructions:
- Provide the EXACT official gene symbol (HUGO Gene Nomenclature Committee approved symbol)
- Choose the most direct, standard gene symbol for the protein
- Avoid gene aliases or alternative names (e.g., prefer "TP53" over "P53")
- Return the primary, commonly used gene symbol
- Gene symbols are typically short, uppercase abbreviations
- Examples: 
  - "p53" → "TP53"
  - "Breast cancer type 1 susceptibility protein" → "BRCA1"
  - "Insulin" → "INS"
- If the protein name is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "gene_symbol": "<gene_symbol_here>"
}}

Protein Name: {protein_name}"""

    response_text = query_together(prompt)
    result = extract_gene_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "protein_name": protein_name,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("protein_to_gene_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("gene_symbol", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_gene_symbol, new_gene_symbol):
    """
    Calculate match result between original and new gene symbols
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_gene_symbol) or pd.isna(new_gene_symbol):
        return 0
    return int(str(original_gene_symbol).strip() == str(new_gene_symbol).strip())

# Main processing function
def main():
    print("Starting protein name to gene symbol mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("protein_to_gene_api_responses_finetuned.jsonl"):
        with open("protein_to_gene_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample protein name
    print("Testing API connection...")
    test_result = get_gene_symbol_from_protein("p53")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the protein names CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_gene_symbol"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            protein_name = row["protein_name"]  # Using the protein_name column from the CSV
            original_gene_symbol = row["GN"]  # Original gene symbol for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {protein_name}")

            start_time = time.time()
            new_gene_symbol = get_gene_symbol_from_protein(protein_name)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new gene symbol
            df.loc[idx, "new_gene_symbol"] = new_gene_symbol
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_gene_symbol, new_gene_symbol)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_gene_symbol}, New: {new_gene_symbol}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_protein_to_gene_results_progress10.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} protein names.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_gene_symbol"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_protein_to_gene_results_final10.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Protein to Gene) ===")
        print(f"Total protein names processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_gene_symbol"] == "None").sum()
        error_results = (df["new_gene_symbol"] == "Error").sum()
        parse_error_results = (df["new_gene_symbol"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid gene symbols returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_gene_symbol"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about bins if available
        if 'bin' in df.columns:
            print(f"\nBreakdown by Bin:")
            bin_stats = df.groupby('bin').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(bin_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_protein_to_gene_results_final5.csv")
    print(f"Progress file: finetuned_protein_to_gene_results_progress5.csv")
    print(f"Log file: protein_to_gene_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_protein, normalized_gene_name, Match_GN")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

5eps

In [1]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_MpajHfgg8br5MUQBjyoc2xmUrzc8mKwHlmo0TXvXH7s"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-GN-8B-PN-GN-5-97fb69a2",  # Your fine-tuned protein model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in gene nomenclature and protein names. Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since gene symbols are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"gene_symbol": "Error"})

# Improved function to extract JSON from the LLM response for gene data
def extract_gene_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'gene_symbol' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'gene_symbol' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"gene_symbol"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'gene_symbol' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract gene_symbol value directly
        gene_symbol_match = re.search(r'"gene_symbol"[\s:]*"([^"]+)"', response_text)
        if gene_symbol_match:
            return {"gene_symbol": gene_symbol_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        gene_symbol_match = re.search(r'gene_symbol["\':\s]+([^"\'}\s,]+)', response_text)
        if gene_symbol_match:
            return {"gene_symbol": gene_symbol_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"gene_symbol": "None"}

        # If all parsing attempts fail
        return {"gene_symbol": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"gene_symbol": "Parse_Error"}

# Enhanced prompt to get gene symbol for a given protein name
def get_gene_symbol_from_protein(protein_name):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the protein name "{protein_name}", provide the corresponding gene symbol.

Instructions:
- Provide the EXACT official gene symbol (HUGO Gene Nomenclature Committee approved symbol)
- Choose the most direct, standard gene symbol for the protein
- Avoid gene aliases or alternative names (e.g., prefer "TP53" over "P53")
- Return the primary, commonly used gene symbol
- Gene symbols are typically short, uppercase abbreviations
- Examples: 
  - "p53" → "TP53"
  - "Breast cancer type 1 susceptibility protein" → "BRCA1"
  - "Insulin" → "INS"
- If the protein name is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "gene_symbol": "<gene_symbol_here>"
}}

Protein Name: {protein_name}"""

    response_text = query_together(prompt)
    result = extract_gene_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "protein_name": protein_name,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("protein_to_gene_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("gene_symbol", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_gene_symbol, new_gene_symbol):
    """
    Calculate match result between original and new gene symbols
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_gene_symbol) or pd.isna(new_gene_symbol):
        return 0
    return int(str(original_gene_symbol).strip() == str(new_gene_symbol).strip())

# Main processing function
def main():
    print("Starting protein name to gene symbol mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("protein_to_gene_api_responses_finetuned.jsonl"):
        with open("protein_to_gene_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample protein name
    print("Testing API connection...")
    test_result = get_gene_symbol_from_protein("p53")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the protein names CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_gene_symbol"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            protein_name = row["protein_name"]  # Using the protein_name column from the CSV
            original_gene_symbol = row["GN"]  # Original gene symbol for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {protein_name}")

            start_time = time.time()
            new_gene_symbol = get_gene_symbol_from_protein(protein_name)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new gene symbol
            df.loc[idx, "new_gene_symbol"] = new_gene_symbol
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_gene_symbol, new_gene_symbol)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_gene_symbol}, New: {new_gene_symbol}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_protein_to_gene_results_progress5.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} protein names.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_gene_symbol"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_protein_to_gene_results_final5.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Protein to Gene) ===")
        print(f"Total protein names processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_gene_symbol"] == "None").sum()
        error_results = (df["new_gene_symbol"] == "Error").sum()
        parse_error_results = (df["new_gene_symbol"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid gene symbols returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_gene_symbol"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about bins if available
        if 'bin' in df.columns:
            print(f"\nBreakdown by Bin:")
            bin_stats = df.groupby('bin').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(bin_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_protein_to_gene_results_final5.csv")
    print(f"Progress file: finetuned_protein_to_gene_results_progress5.csv")
    print(f"Log file: protein_to_gene_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_protein, normalized_gene_name, Match_GN")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting protein name to gene symbol mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Test result: TP53
API test successful, proceeding with batch processing...
Loaded 3978 rows from C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv
Columns in the dataset: ['ID', 'AC', 'GN', 'protein_name', 'GO_F Count', 'GO_P Count', 'GO_C Count', 'go_counts', 'bin', 'normalized_protein', 'normalized_gene_name', 'Match_GN', 'pmc_GN', 'PMC_protein_name']
Processing 1/3978: Exonuclease 3'-5' domain-containing protein 2 
Sending request (attempt 1/5)...
  Original: EXD2, New: EXO3, Match: 0
Progress saved. Processed 1/3978 protein names.
Waiting 2.68 seconds before next request...
Processing 2/3978: Exosome complex component MTR3
Sending request (attempt 1/5)...
  Original: EXOSC6, New: EXOSC3, Match: 0
Waiting 3.37 seconds before next request...
Processing 3/3978: Innate immunity acti

1 ep

In [3]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-GN-8B-GN-PN-1-426db02a",  # Your fine-tuned protein model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in gene nomenclature and protein names. Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since gene symbols are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"gene_symbol": "Error"})

# Improved function to extract JSON from the LLM response for gene data
def extract_gene_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'gene_symbol' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'gene_symbol' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"gene_symbol"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'gene_symbol' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract gene_symbol value directly
        gene_symbol_match = re.search(r'"gene_symbol"[\s:]*"([^"]+)"', response_text)
        if gene_symbol_match:
            return {"gene_symbol": gene_symbol_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        gene_symbol_match = re.search(r'gene_symbol["\':\s]+([^"\'}\s,]+)', response_text)
        if gene_symbol_match:
            return {"gene_symbol": gene_symbol_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"gene_symbol": "None"}

        # If all parsing attempts fail
        return {"gene_symbol": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"gene_symbol": "Parse_Error"}

# Enhanced prompt to get gene symbol for a given protein name
def get_gene_symbol_from_protein(protein_name):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the protein name "{protein_name}", provide the corresponding gene symbol.

Instructions:
- Provide the EXACT official gene symbol (HUGO Gene Nomenclature Committee approved symbol)
- Choose the most direct, standard gene symbol for the protein
- Avoid gene aliases or alternative names (e.g., prefer "TP53" over "P53")
- Return the primary, commonly used gene symbol
- Gene symbols are typically short, uppercase abbreviations
- Examples: 
  - "p53" → "TP53"
  - "Breast cancer type 1 susceptibility protein" → "BRCA1"
  - "Insulin" → "INS"
- If the protein name is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "gene_symbol": "<gene_symbol_here>"
}}

Protein Name: {protein_name}"""

    response_text = query_together(prompt)
    result = extract_gene_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "protein_name": protein_name,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("protein_to_gene_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("gene_symbol", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_gene_symbol, new_gene_symbol):
    """
    Calculate match result between original and new gene symbols
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_gene_symbol) or pd.isna(new_gene_symbol):
        return 0
    return int(str(original_gene_symbol).strip() == str(new_gene_symbol).strip())

# Main processing function
def main():
    print("Starting protein name to gene symbol mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("protein_to_gene_api_responses_finetuned.jsonl"):
        with open("protein_to_gene_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample protein name
    print("Testing API connection...")
    test_result = get_gene_symbol_from_protein("p53")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the protein names CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_gene_symbol"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            protein_name = row["protein_name"]  # Using the protein_name column from the CSV
            original_gene_symbol = row["GN"]  # Original gene symbol for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {protein_name}")

            start_time = time.time()
            new_gene_symbol = get_gene_symbol_from_protein(protein_name)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new gene symbol
            df.loc[idx, "new_gene_symbol"] = new_gene_symbol
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_gene_symbol, new_gene_symbol)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_gene_symbol}, New: {new_gene_symbol}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_protein_to_gene_results_progress1.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} protein names.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_gene_symbol"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_protein_to_gene_results_final1.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Protein to Gene) ===")
        print(f"Total protein names processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_gene_symbol"] == "None").sum()
        error_results = (df["new_gene_symbol"] == "Error").sum()
        parse_error_results = (df["new_gene_symbol"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid gene symbols returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_gene_symbol"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about bins if available
        if 'bin' in df.columns:
            print(f"\nBreakdown by Bin:")
            bin_stats = df.groupby('bin').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(bin_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_protein_to_gene_results_final5.csv")
    print(f"Progress file: finetuned_protein_to_gene_results_progress5.csv")
    print(f"Log file: protein_to_gene_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_protein, normalized_gene_name, Match_GN")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting protein name to gene symbol mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Test result: TP53
API test successful, proceeding with batch processing...
Loaded 3978 rows from C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv
Columns in the dataset: ['ID', 'AC', 'GN', 'protein_name', 'GO_F Count', 'GO_P Count', 'GO_C Count', 'go_counts', 'bin', 'normalized_protein', 'normalized_gene_name', 'Match_GN', 'pmc_GN', 'PMC_protein_name']
Processing 1/3978: Exonuclease 3'-5' domain-containing protein 2 
Sending request (attempt 1/5)...
  Original: EXD2, New: EXO2, Match: 0
Progress saved. Processed 1/3978 protein names.
Waiting 2.43 seconds before next request...
Processing 2/3978: Exosome complex component MTR3
Sending request (attempt 1/5)...
  Original: EXOSC6, New: MTR3, Match: 0
Waiting 3.43 seconds before next request...
Processing 3/3978: Innate immunity activa